## update anno

update the annotation json files if any

only json, no images

put all json under folder named 'new_anno' in the same path of this cookbook

In [15]:
from util import *
import os

folder_path = "new_anno"

# Get a list of all files in the folder
files_list = os.listdir(folder_path)
for new_anno_name in files_list:
    folder = new_anno_name.split('.')[0]
    file_path = folder_path + '/' + new_anno_name
    new_path = 'data_src/' + folder + '/annotations.json'
    copy_file(file_path, new_path)

# (merge annos) create train and val dataset

create train and val dataset from the kenametal dataset

kenametal dataset:

put kenametal dataset named 'data_src' in the same path of this cookbook

data_merge_side and data_merge_top will be created

- dataset.ipynb  # this cookbook
- data_merge_side # merge coco for side
- data_merge_top # for top
- data_src
  - folders
    - annotations.json
    - images
    - visualized (no use)



dataset
  - annotations
    - train
    - val
  - train2017
    - images
  - val2017
    - images

In [7]:
from util import *

In [17]:
# Example usage
directory_path = 'data_src/'
prefix = 'side'
folder_names = get_folders_with_prefix(directory_path, prefix)

In [18]:
side_folders = ['side_Video_20250304224204468',
 'side_Video_20250304220951233',
 'side_Video_20250304221925029',
 'side_Video_20250304222515804',
 'side_Video_20250304220325292',
 'side_Video_20250304223058494',
 'side_Video_20250304223625443']

top_folders = ['top_Video_20250304191858967',
 'top_Video_20250304195342288',
 'top_Video_20250304194042559',
 'top_Video_20250304181532896',
 'top_Video_20250304184757847',
 'top_Video_20250304180336961',
 'top_Video_20250304192819030',
 'top_Video_20250304182609820',
 'top_Video_20250304185856120',
 'top_Video_20250304190913888']

In [2]:
side_folder_train, side_folder_test = ['side_Video_20250304224204468',
 'side_Video_20250304222515804',
 'side_Video_20250304220325292',
 'side_Video_20250304223058494',
 'side_Video_20250304223625443'],  ['side_Video_20250304221925029', 'side_Video_20250304220951233',]

top_folder_train, top_folder_test = ['top_Video_20250304191858967',
 'top_Video_20250304195342288',
 'top_Video_20250304194042559',
 'top_Video_20250304181532896',
 'top_Video_20250304184757847',
 'top_Video_20250304180336961',
 'top_Video_20250304182609820',
 'top_Video_20250304185856120',
 'top_Video_20250304190913888'], ['top_Video_20250304192819030']

In [9]:
def coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=True):
    '''
      merge multiple coco json into one
      merge annotations
      copy images
      with re-index on image id and annotation id
      
      coco_list: target annotation list
      base_coco: base info about dataset
      dst_folder: destination folder
    '''
    
    img_idx = 0
    anno_idx = 0
    img_list = []
    anno_list = []
    #make_folder(dst_folder + 'annotations/')

    if train:
        #train
        dst_anno = dst_folder + 'annotations/custom_train.json'
        dst_img_folder = dst_folder + 'train2017/'
        dst_img_gt_folder = gt_folder + 'train2017_gt/'
    else:
        #test
        dst_anno = dst_folder + 'annotations/custom_val.json'
        dst_img_folder = dst_folder + 'val2017/'
        dst_img_gt_folder = gt_folder + 'val2017_gt/'

    make_folder(dst_img_folder)
    make_folder(dst_img_gt_folder)

    for i in coco_list:
        tmp_folder = 'data_src/' + i
        tmp_img_folder = tmp_folder + '/images/'
        tmp_img_gt_folder = tmp_folder + '/visualized/'
        tmp_anno = tmp_folder + '/annotations.json'

        # img re-index
        anno = load(tmp_anno)
        tmp_img_index_dict = {} #old and new pair, image id
        for img in anno['images']:
            tmp_img_index_dict[img['id']] = img_idx
            # update image id
            img['id'] = img_idx
            # update file name
            tmp_img = tmp_img_folder + img['file_name']
            tmp_img_gt = tmp_img_gt_folder + img['file_name']
            img['file_name'] = str(img_idx) + '.png'
            # copy image
            tmp_img_new = dst_img_folder + img['file_name']
            tmp_img_gt_new = dst_img_gt_folder + img['file_name']
            copy_file(tmp_img, tmp_img_new)
            copy_file(tmp_img_gt, tmp_img_gt_new)
            
            # append to img list
            img_list.append(img)
            
            img_idx += 1
        
        # update annotations
        for an_box in anno['annotations']:
            # update an_box be integer
            an_box['category_id'] = int(an_box['category_id'])
            an_box['area'] = int(an_box['area'])
            an_box['iscrowd'] = int(an_box['iscrowd'])
            an_box['bbox'] = [int(i) for i in an_box['bbox']]
            an_box['segmentation'] = {
                'counts': [int(i) for i in an_box['segmentation']['counts']],
                'size': [int(i) for i in an_box['segmentation']['size']]
            }
            
            
            # update image id
            an_box['image_id'] = tmp_img_index_dict[an_box['image_id']]
            # update id
            an_box['id'] = anno_idx
            # update segmentation
            #an_box['segmentation'] = [an_box['segmentation']['counts']]
            
            # append to annotation list
            anno_list.append(an_box)
            
            anno_idx += 1
      
      
    base_coco['images'] = img_list
    base_coco['annotations'] = anno_list
    # save to json
    save(dst_anno, base_coco)

In [10]:
base_coco = {
  "licenses": [
    {
      "name": "",
      "id": 0,
      "url": ""
    }
  ],
  "info": {
    "contributor": "",
    "date_created": "",
    "description": "",
    "url": "",
    "version": "",
    "year": ""
  },
  "categories": [
    {
      "id": 1,
      "name": "flaking",
      "supercategory": ""
    }
  ]
}
dst_folder = 'data_merge_side/'
gt_folder = 'gt/side/'
#make_folder(dst_folder)
coco_list = side_folder_train
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=True)
coco_list = side_folder_test
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=False)

dst_folder = 'data_merge_top/'
gt_folder = 'gt/top/'
#make_folder(dst_folder)
coco_list = top_folder_train
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=True)
coco_list = top_folder_test
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=False)

# get negative (non-flaking) case data, only from validation video

put validation video under folder 'src_vid'

data_neg folder will be created

- dataset.ipynb  # this cookbook
- src_vid
  - videos
- data_neg
  - side1  #first side val video
  - side2  #second
  - top

In [4]:
from moviepy import VideoFileClip, TextClip, CompositeVideoClip
def cut_video(input_video, output_video, start_time, end_time):
    clip = (
        VideoFileClip(input_video)
        .subclipped(start_time, end_time) #between in seconds
        .with_volume_scaled(0)
    )

    final_video = CompositeVideoClip([clip])
    final_video.write_videofile(output_video)
#cut_video("data/1.mp4", "data/1_cut.mp4", 0, 10)

#### convert video to images
import cv2
def convert_video_to_images(input_video, output_folder):
    # Create output folder if it doesn't exist
    output_folder = make_folder(output_folder)

    # Open the video file
    video_capture = cv2.VideoCapture(input_video)
    success, frame = video_capture.read()
    count = 0

    # Read each frame and save it as an image
    while success: # and count < 1000:
        image_path = os.path.join(output_folder, f"frame_{count:d}.jpg")  # Adjust the format as per your requirement
        cv2.imwrite(image_path, frame)  # Save the frame as an image
        success, frame = video_capture.read()  # Read next frame
        count += 1

    # Release the video capture object
    video_capture.release()
    return count #the number of frames saved (good/successful frames, not the whole frame count)

In [13]:
side_1 = 'src_vid/Video_20250304221925029.avi'
side_2 = 'src_vid/Video_20250304220951233.avi'
top = 'src_vid/Video_20250304192819030.avi'

In [8]:
convert_video_to_images(side_1, "src_vid/val_cut_side1/")

842

In [9]:
import os
import random
import shutil

def copy_random_files(source_folder, destination_folder, ranges, num_files):
    # Create the destination folder if it doesn't exist
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    files = os.listdir(source_folder)

    selected_files = []
    for r in ranges:
        start_range, end_range = r
        #files_range = [file for file in files if file.startswith("frame_") and file.endswith(".jpg")]
        files_range = [file for file in files if int(file.split("_")[1].split(".")[0]) >= start_range and int(file.split("_")[1].split(".")[0]) <= end_range]
        
        selected_files.extend(files_range)

    # Randomly select files from each range
    random_files = random.sample(selected_files, min(num_files, len(selected_files)))

    # Copy selected files to the destination folder
    for file in random_files:
        source_file = os.path.join(source_folder, file)
        destination_file = os.path.join(destination_folder, file)
        shutil.copyfile(source_file, destination_file)

# Example usage:
source_folder = 'src_vid/val_cut_side1/'
destination_folder = "data_neg/side1/"
ranges = [[300, 840]]
num_files = 40

copy_random_files(source_folder, destination_folder, ranges, num_files)

In [10]:
convert_video_to_images(side_2, "src_vid/val_cut_side2/")

860

In [11]:
source_folder = 'src_vid/val_cut_side2/'
destination_folder = "data_neg/side2/"
ranges = [[70, 850]]
num_files = 100

copy_random_files(source_folder, destination_folder, ranges, num_files)

In [14]:
convert_video_to_images(top, "src_vid/val_cut_top/")

1644

In [15]:
source_folder = 'src_vid/val_cut_top/'
destination_folder = "data_neg/top/"
ranges = [[0, 50], [440, 880], [1250, 1640]]
num_files = 60

copy_random_files(source_folder, destination_folder, ranges, num_files)

# ground truth visualization (update visual)

create visualization ground truth, bounding box and segmentation based on coco annotation (merged coco)

a new folder called gt_cust will be created containing all images

- dataset.ipynb  # this cookbook
- data_merge_side
- data_merge_top
- gt_cust
  - side 
    - train2017
    - val2017
  - top
    - train2017
    - val2017

In [4]:
%matplotlib inline
import pycocotools.coco as coco
from pycocotools.coco import COCO
import numpy as np
import skimage.io as io
import matplotlib.pyplot as plt
import pylab
pylab.rcParams['figure.figsize'] = (10.0, 8.0)

def gen_gt(coco_folder, image_folder_name, output_folder, start_idx, end_idx):
    
    '''
    coco_folder: the folder name of coco dataset, data_merge_side
    image_folder_name: the folder name of img, train2017
    output_folder: the folder name of output, gt_cust/side/train2017/
    '''

    data_path = '/home/loke_tictag_io/test/ken/'
    dataDir=f'{data_path}{coco_folder}/'
    #dataDir=f'{data_path}data_merge_side/'
    #dataType='train2017'
    dataType=image_folder_name
    if dataType == 'train2017':
        annFile='{}annotations/custom_train.json'.format(dataDir)
    else:
        annFile='{}annotations/custom_val.json'.format(dataDir)

    # initialize COCO api for instance annotations
    coco=COCO(annFile)
    # display COCO categories and supercategories
    #cats = coco.loadCats(coco.getCatIds())
    # load and display image
    catIds = coco.getCatIds(catNms=['flaking']);
    imgIds = coco.getImgIds(catIds=catIds );
    
    if start_idx == None:
        start_idx = 0
    
    if end_idx == None:
        end_idx = len(imgIds)

    #for i in range(0, len(imgIds)):
    for i in range(start_idx, end_idx):
        img_id = imgIds[i]
        img = coco.loadImgs(img_id)[0]

        img_name = '%s/%s/%s'%(dataDir, dataType, img['file_name'])
        #print('Image name: {}'.format(img_name))

        I = io.imread(img_name)
        plt.figure()

        annIds = coco.getAnnIds(imgIds=img['id'], catIds=catIds)
        anns = coco.loadAnns(annIds)

        # Display your image and annotations without axis
        plt.imshow(I)
        coco.showAnns(anns, draw_bbox=True)
        plt.axis('off')  # Turn off axis

        # Save the current figure without the axis
        plt.savefig(f'{output_folder}{i}.png', bbox_inches='tight', pad_inches=0)
        plt.close()

cv2 to visualize

In [35]:
%matplotlib inline
import pycocotools.coco as coco
from pycocotools.coco import COCO
import numpy as np
import skimage.io as io
import matplotlib.pyplot as plt
import pylab
import cv2
from PIL import Image
from pycocotools import mask

pylab.rcParams['figure.figsize'] = (10.0, 8.0)

def gen_gt_cv2(coco_folder, image_folder_name, output_folder, start_idx, end_idx):
    
    '''
    coco_folder: the folder name of coco dataset, data_merge_side
    image_folder_name: the folder name of img, train2017
    output_folder: the folder name of output, gt_cust/side/train2017/
    '''

    data_path = '/home/loke_tictag_io/test/ken/'
    dataDir=f'{data_path}{coco_folder}/'
    #dataDir=f'{data_path}data_merge_side/'
    #dataType='train2017'
    dataType=image_folder_name
    if dataType == 'train2017':
        annFile='{}annotations/custom_train.json'.format(dataDir)
    else:
        annFile='{}annotations/custom_val.json'.format(dataDir)

    # initialize COCO api for instance annotations
    coco=COCO(annFile)
    # display COCO categories and supercategories
    #cats = coco.loadCats(coco.getCatIds())
    # load and display image
    catIds = coco.getCatIds(catNms=['flaking']);
    imgIds = coco.getImgIds(catIds=catIds );
    
    if start_idx == None:
        start_idx = 0
    
    if end_idx == None:
        end_idx = len(imgIds)

    #for i in range(0, len(imgIds)):
    for i in range(start_idx, end_idx):
        img_id = imgIds[i]
        img = coco.loadImgs(img_id)[0]

        img_name = '%s/%s/%s'%(dataDir, dataType, img['file_name'])
        #print('Image name: {}'.format(img_name))

        annIds = coco.getAnnIds(imgIds=img['id'], catIds=catIds)
        anns = coco.loadAnns(annIds)
        
        #drwa
        boxes = anns[0]['bbox']

        def mask_from_rle_ann(segmentation):
            rle = mask.frPyObjects(segmentation, segmentation["size"][0], segmentation["size"][1])
            binary_mask = mask.decode(rle)
            return binary_mask
        eg_mask = mask_from_rle_ann(anns[0]['segmentation'])

        im = Image.open(img_name)
        cv_img = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2BGR)

        # Convert the mask to a binary image
        mask_binary = (eg_mask > 0).astype(np.uint8) * 255

        # Find contours of the mask
        #contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Draw the contours on the image
        #cv2.drawContours(cv_img, contours, -1, (50, 50, 255), 3)  # Draw contour with specified color and thickness

        alpha = 0.8  # Transparency level
        mask_rgb = cv2.merge([np.zeros_like(mask_binary), np.zeros_like(mask_binary), mask_binary])
        cv_img = cv2.addWeighted(cv_img, alpha, mask_rgb, 1 - alpha, 0)

        # Draw rectangles and text on the image
        if boxes is not None:
            for xmin, ymin, w, h in [boxes]:
                xmax = xmin + w
                ymax = ymin + h
                pt1 = (int(xmin), int(ymin))
                pt2 = (int(xmax), int(ymax))
                cv2.rectangle(cv_img, pt1, pt2, (255, 246, 132), 10)  # Blue color and larger thickness

        # Save the image
        cv2.imwrite(f'{output_folder}{i}.png', cv_img)

In [ ]:
gen_gt_cv2('data_merge_side', 'train2017', 'gt_cust/side/train2017/', None, None) #96 images
gen_gt_cv2('data_merge_side', 'val2017', 'gt_cust/side/val2017/', None, None) #31
gen_gt_cv2('data_merge_top', 'train2017', 'gt_cust/top/train2017/', None, None) #176
gen_gt_cv2('data_merge_top', 'val2017', 'gt_cust/top/val2017/', None, None) #57

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


# top-view, merge, setting different classes

- dataset.ipynb  # this cookbook
- data_merge_top_mc # for top, multiple classes
- data_src
  - folders
    - annotations.json
    - images
    - visualized (no use)

In [ ]:
from util import *

top_folder_train, top_folder_test = ['top_Video_20250304191858967',
 'top_Video_20250304195342288',
 'top_Video_20250304194042559',
 'top_Video_20250304181532896',
 'top_Video_20250304184757847',
 'top_Video_20250304180336961',
 'top_Video_20250304182609820',
 'top_Video_20250304185856120',
 'top_Video_20250304190913888'], ['top_Video_20250304192819030']
# color is a way of missing, missing in terms of thin layer of surface
label_train = ['missing', 'edge', 'edge', 'color', 'edge', 'edge', 'color', 'edge', 'color']
label_id_train = [3, 2, 2, 1, 2, 2, 1, 2, 1]
label_test = ['color'] #or missing
label_id_test = [1]

In [3]:
def coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=True):
    '''
      merge multiple coco json into one
      merge annotations
      copy images
      with re-index on image id and annotation id
      
      coco_list: target annotation list
      base_coco: base info about dataset
      dst_folder: destination folder
    '''
    label_id_train = [3, 2, 2, 1, 2, 2, 1, 2, 1]
    label_id_test = [1]
    img_idx = 0
    anno_idx = 0
    img_list = []
    anno_list = []
    #make_folder(dst_folder + 'annotations/')

    if train:
        #train
        dst_anno = dst_folder + 'annotations/custom_train.json'
        dst_img_folder = dst_folder + 'train2017/'
        dst_img_gt_folder = gt_folder + 'train2017_gt/'
        label_list = label_id_train
    else:
        #test
        dst_anno = dst_folder + 'annotations/custom_val.json'
        dst_img_folder = dst_folder + 'val2017/'
        dst_img_gt_folder = gt_folder + 'val2017_gt/'
        label_list = label_id_test

    make_folder(dst_img_folder)
    make_folder(dst_img_gt_folder)

    for i in range(len(coco_list)):
        label = label_list[i]
        anno_folder = coco_list[i]
        tmp_folder = 'data_src/' + anno_folder
        tmp_img_folder = tmp_folder + '/images/'
        tmp_img_gt_folder = tmp_folder + '/visualized/'
        tmp_anno = tmp_folder + '/annotations.json'

        # img re-index
        anno = load(tmp_anno)
        tmp_img_index_dict = {} #old and new pair, image id
        for img in anno['images']:
            tmp_img_index_dict[img['id']] = img_idx
            # update image id
            img['id'] = img_idx
            # update file name
            tmp_img = tmp_img_folder + img['file_name']
            tmp_img_gt = tmp_img_gt_folder + img['file_name']
            img['file_name'] = str(img_idx) + '.png'
            # copy image
            tmp_img_new = dst_img_folder + img['file_name']
            tmp_img_gt_new = dst_img_gt_folder + img['file_name']
            copy_file(tmp_img, tmp_img_new)
            copy_file(tmp_img_gt, tmp_img_gt_new)
            
            # append to img list
            img_list.append(img)
            
            img_idx += 1
        
        # update annotations
        for an_box in anno['annotations']:
            # update an_box be integer
            an_box['category_id'] = int(label)
            an_box['area'] = int(an_box['area'])
            an_box['iscrowd'] = int(an_box['iscrowd'])
            an_box['bbox'] = [int(i) for i in an_box['bbox']]
            an_box['segmentation'] = {
                'counts': [int(i) for i in an_box['segmentation']['counts']],
                'size': [int(i) for i in an_box['segmentation']['size']]
            }
            
            
            # update image id
            an_box['image_id'] = tmp_img_index_dict[an_box['image_id']]
            # update id
            an_box['id'] = anno_idx
            # update segmentation
            #an_box['segmentation'] = [an_box['segmentation']['counts']]
            
            # append to annotation list
            anno_list.append(an_box)
            
            anno_idx += 1
      
      
    base_coco['images'] = img_list
    base_coco['annotations'] = anno_list
    # save to json
    save(dst_anno, base_coco)

In [5]:
base_coco = {
  "licenses": [
    {
      "name": "",
      "id": 0,
      "url": ""
    }
  ],
  "info": {
    "contributor": "",
    "date_created": "",
    "description": "",
    "url": "",
    "version": "",
    "year": ""
  },
  "categories": [
    {
      "id": 1,
      "name": "color",
      "supercategory": "flaking"
    },
    {
      "id": 2,
      "name": "edge",
      "supercategory": "flaking"
    },
    {
      "id": 3,
      "name": "missing",
      "supercategory": "flaking"
    }
        
  ]
}


dst_folder = 'data_merge_top_mc/'
gt_folder = 'gt/top/'
#make_folder(dst_folder)
coco_list = top_folder_train
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=True)
coco_list = top_folder_test
coco_merge(coco_list, base_coco, dst_folder, gt_folder, train=False)

# side-view, merge, setting different classes

- dataset.ipynb  # this cookbook
- data_merge_side_mc # for top, multiple classes
- data_src
  - folders
    - annotations.json
    - images
    - visualized (no use)

In [ ]:
from util import *

side_folder_train, side_folder_test = ['side_Video_20250304224204468',
 'side_Video_20250304222515804',
 'side_Video_20250304220325292',
 'side_Video_20250304223058494',
 'side_Video_20250304223625443'],  ['side_Video_20250304221925029', 'side_Video_20250304220951233',]

#edge, color: 2, 1
label_train = ['color', 'edge', 'color', 'color', 'color']
label_id_train = [1, 2, 1, 1, 1]
#5029 have missing first 8 and color next 12 (total 20)
label_test = ['color', 'color']
label_id_test = [1, 1]

only one train video could treated separately, no mc for now